<center>
  <h3>More than a Hack 2027 &middot; Phase 1</h3>
  <h2>Frame-level Ball Possession (Final Method and Explainability)</h2>

  <p><strong>Metrica Sports &times; FC Barcelona Innovation Hub &times; Mobile World Capital</strong></p>
</center>

---

### Carried By Admin Team - Sepuluh Nopember Institute of Technology - Indonesia

- **Rasya Arya Ramadhan**
- **Benedictus Ryu Gunawan**
- **Ahmad Naufal Farras**
- **Jeremy Mattathis Mboe**
- **Jylan Annisa Mumtaza Syidana**

---

## Summary

This notebook reruns the team's **final method**, `farras_wave5_short_gru25_dead45` (registry [E06](CARD.md)), on the target match and explains how it reaches each decision. It labels every one of the 199,319 tactical frames of FC Barcelona vs Athletic Club as `H`, `A`, `D` or `X`.

It has three parts:

1. **Training from scratch** (`RETRAIN = True`, the default). It downloads the public training data (IDSSE, SkillCorner), builds the training packs and features, trains the set-sequence BiGRU and the short-loss PP-TCN, then builds the target inputs from the licensed match bundle in `data/`. Every step calls the campaigns' own code through [`train_from_scratch.py`](train_from_scratch.py). With `RETRAIN = False` the notebook uses existing checkpoints and caches instead.
2. **Final run.** Inference, fusion and decoding through the campaign's own functions, then the CSV and its SHA-256 compared with the locked artifact in [`docs/final_method.json`](../../docs/final_method.json). A freshly trained model can differ from the frozen one (other hardware or library versions), so a different hash after retraining means *retrained*, not *broken*; the public scores show how close it is.
3. **Explainability.** Eight lenses, from the outside in: what each pipeline stage does to one decision, which stages matter (counterfactual ablations), which inputs the network uses (Integrated Gradients), how far in time it looks, which players the set encoder attends to, what happens when an input group is removed, where the three seeds disagree, and where the errors on the public window come from.

Nothing in this notebook trains, tunes or selects anything. The public 5-minute ground truth (GT) is used **only** for post-hoc diagnostics, as in the campaign (see `RULES.md`). The ranking metric is spellF1:

$$
\mathrm{IoU}(p, g) = \frac{|p \cap g|}{|p \cup g|},
\qquad
\mathrm{spellF1} = \frac{1}{5}\sum_{\tau \in \{0.3,\,0.4,\,0.5,\,0.6,\,0.7\}} F1_\tau
$$

# Methodology

**Public training data → train BiGRU + short-loss PP-TCN → target inputs → PP-TCN inference (3 seeds × 2 team views) → 75/25 joint mix with the BiGRU → `dead45` decoder → 25-fps CSV → hash check → explanations.**

The final method, as locked in [`docs/FINAL_METHOD.md`](../../docs/FINAL_METHOD.md):

| Stage | What it does |
|---|---|
| Observations | Prepared automatic-tracking features at 5 fps: 23 team-shape channels, 1 validity flag and 5 ball channels per team view, plus a set of up to 48 per-player vectors (10 features each). |
| Short-loss PP-TCN | Residual dilated TCN (64 channels, kernel 5, dilations 1/2/4/8/16/1) with an attention + max pooled player-set branch. Two logits per sample: dead ball and "my team has the ball". Trained with the state loss doubled on complete live spells < 4 s. Three seeds, two mirrored views, averaged. |
| Frozen BiGRU (wave3) | Player-set encoder + 2-layer bidirectional GRU. Its probabilities are cached; it is not retrained. |
| Fusion | Joint $p(H), p(A), p(D)$: $0.75\,p_\text{TCN} + 0.25\,p_\text{GRU}$, renormalised. |
| Decoder `dead45` | Dead mask: $p(D) > 0.45$ for ≥ 1 s. Team evidence: $\kappa \log p_T$ plus a logistic proximity term when the ball is visible and a player is within 0.35 m. Two-state Viterbi reset in every live interval; switch cost λ chosen so the possession count matches the training possession rate. Live runs < 1 s merged. |

Explainability methods, and the question each answers:

| Section | Method | Question |
|---|---|---|
| [9] | Stage-by-stage trace | How does one decision travel through the pipeline? |
| [10] | Counterfactual ablation | Which stage changes the output most? |
| [11] | Integrated Gradients | Which input channels drive the dead and team logits? |
| [12] | Input-gradient profile | How far back and forward in time does the TCN look? |
| [13] | Attention read-out | Which players does the set encoder weight? |
| [14] | Feature-group occlusion | What breaks when an input group is removed? |
| [15] | Seed disagreement | Where is the model uncertain? |
| [16] | Error anatomy | Which public-window errors remain, and why? |

All figures share the E00 palette: `#e94f37` (home / primary), `#393e41` (away / secondary), `#f6f7eb` (background).

# Environment

Run from inside the repository (numbered layout, `experiments/E06_wave5/`). Training the PP-TCN takes 3 seeds × 1,500 updates and is meant for a GPU (it also runs on CPU, much slower); the BiGRU trains on CPU with one thread, as in its campaign. Inference and everything after it run on CPU. Training downloads about 4.5 GB of public data (IDSSE ≈ 2.6 GB, SkillCorner ≈ 1.8 GB). The target steps need the licensed match bundle unpacked in `data/` ([DATA.md](../../DATA.md)); keep them on the licensed machine.

With `RETRAIN = False`, the inputs below must already exist. They are **gitignored**, so they are not in a fresh clone of either branch; without them section [1] stops with `FileNotFoundError` and a list of the missing paths (this is the expected "blocked: missing inputs" outcome, not a bug):

| Artifact | Numbered layout (main) | Original layout (farras branch, release `7254990`) |
|---|---|---|
| Target observations | `experiments/E04_wave3/cache/data/target_clean.npz` | `experiments/exp_farras/wave3/cache/data/target_clean.npz` |
| Target features | `experiments/E05_wave4/cache/features/target_clean.npz` | `experiments/exp_farras/wave4/cache/features/target_clean.npz` |
| Frozen BiGRU probabilities | `experiments/E05_wave4/cache/prior/target_clean.npz` | `experiments/exp_farras/wave4/cache/prior/target_clean.npz` |
| Short-loss TCN checkpoint | `experiments/E06_wave5/out/models/short_tcn.pt` | `experiments/exp_farras/wave5/out/models/short_tcn.pt` |
| 19 training packs (rate only) | `E04_wave3/cache/data/J03*_r0.npz`, `E05_wave4/cache/data/skillcorner_*_clean.npz` | same files under `exp_farras/wave3/` and `exp_farras/wave4/` |

The notebook detects the layout itself; the cells below print the exact paths they look for. With `RETRAIN = True`, sections [T1]–[T4] produce all of them. The training part can also run on a GPU machine with `python experiments/E06_wave5/train_from_scratch.py public`, and the target part on the licensed machine with `... target`; copy the two checkpoints (`E04_wave3/out/models/hybrid.pt`, `E06_wave5/out/models/short_tcn.pt`) and the 19 training packs between them.

The checkpoint is verified against the SHA-256 recorded at selection time in `results/wave5_development.json`. Dependencies: `experiments/E06_wave5/requirements.txt`.

In [ ]:
from pathlib import Path
import sys, os, json, hashlib, math, time, warnings

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "games.py").exists())
EXP = ROOT / "experiments"
# Works in both repository layouts: numbered folders on main, or the original layout of the
# farras branch / release commit 7254990 (experiments/exp_farras/wave5 + exp_farras/results).
LAYOUTS = [(EXP / "E06_wave5", EXP / "E06_wave5" / "results"),
           (EXP / "exp_farras" / "wave5", EXP / "exp_farras" / "results")]
HERE, RESULTS = next(((h, r) for h, r in LAYOUTS if (h / "support.py").exists()), (None, None))
if HERE is None:
    raise FileNotFoundError("wave5 campaign code not found in experiments/E06_wave5 or experiments/exp_farras/wave5")
for p in (ROOT / "evaluation", HERE):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))
os.environ.setdefault("FARRAS_TRAIN_DEVICE", "cpu")
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
warnings.filterwarnings("ignore", category=RuntimeWarning)

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Patch
import numpy as np
import pandas as pd
import torch
from IPython.display import display

import support as c                      # wave5 campaign helpers (paths, loaders, training statistics)
import training_models as models         # short-loss PP-TCN fit/predict
import segment_decoder as decoder        # dead45 and the other wave5 decoders
import adaptive_fusion as fusion         # joint probability mixing
import possession_metrics as pm          # official metric implementation
old_runner = c.import_path("wave5_frozen_wave4_runner", c.PREVIOUS / "run.py")   # CSV writer used by the campaign
FEATS = c.import_path("wave4_feature_names", c.PREVIOUS / "features.py")
torch.set_grad_enabled(False)

# --- palette (same as E00) -------------------------------------------------------------
C1, C2, C3 = "#e94f37", "#393e41", "#f6f7eb"
INK, GREY, WARM = "#393e41", "#a3a7a6", "#393e41"
TEAL = "#1f6f8b"
CMAP = LinearSegmentedColormap.from_list("sunset", [C3, C1, C2])
TEAM = {"H": C1, "A": C2, "D": GREY, "X": "#dcdcd0"}
mpl.rcParams.update({
    "figure.facecolor": C3, "savefig.facecolor": C3, "axes.facecolor": "white",
    "axes.edgecolor": INK, "axes.labelcolor": INK, "text.color": INK,
    "xtick.color": INK, "ytick.color": INK, "axes.titleweight": "bold",
    "axes.titlesize": 10, "axes.labelsize": 9, "xtick.labelsize": 8, "ytick.labelsize": 8,
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": False,
    "font.family": "sans-serif", "legend.frameon": False, "legend.fontsize": 8, "figure.dpi": 110,
    "axes.prop_cycle": mpl.cycler(color=[C1, C2, GREY, TEAL]),
})

def panel(ax, letter):
    ax.text(-0.09, 1.07, letter, transform=ax.transAxes, fontsize=12, fontweight="bold", va="top")

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

FPS25, FPS5 = 25, 5
FINAL = json.loads((ROOT / "docs" / "final_method.json").read_text())
DEV = json.loads((RESULTS / "wave5_development.json").read_text())
SPEC = DEV["selected"]                                   # frozen at selection time
DEC = dict(SPEC["decoder"])
assert SPEC["family"] == "short_gru25" and DEC["name"] == "dead45"
assert {k: v for k, v in DEC.items() if k != "name"} == {k: v for k, v in FINAL["pipeline"]["decoder"].items() if k != "name"}
GRU_W = FINAL["pipeline"]["components"][1]["weight"]

RETRAIN = True                       # False: reuse existing checkpoints/caches instead of training
TRAIN_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
FROZEN_CSV_SHA = FINAL["artifact"]["sha256"]
if RETRAIN:
    import train_from_scratch as tfs
print("ROOT:", ROOT, "| campaign folder:", HERE.relative_to(ROOT))
print("method:", FINAL["method_id"], "| family:", SPEC["family"], "| decoder:", DEC["name"], "| GRU weight:", GRU_W)
print("RETRAIN:", RETRAIN, "| training device:", TRAIN_DEVICE)

# Training from scratch

## [T1] Public training data

Downloads the 7 IDSSE matches (PySport mirror, sizes and LFS SHA-256 verified) and the 20 SkillCorner open-data matches, then builds 5-fps training packs and the PP-TCN features with the campaigns' own preparation code (`E04_wave3/prepare_idsse.py`, `E05_wave4/prepare_skillcorner.py`, `E05_wave4/prepare_features.py`). IDSSE gets three label-free corruption rounds per match. Existing packs are reused; pass `force=True` to rebuild. No challenge data is involved.

In [ ]:
if RETRAIN:
    t0 = time.time()
    tfs.prepare_public_data(force=False)
    print(f"public data ready in {time.time() - t0:.0f} s")

## [T2] Train the set-sequence BiGRU

The 25% component: a player-set encoder followed by a two-layer bidirectional GRU (hidden 32, window 300, 8 epochs, seed 2027), trained on the 5 IDSSE development matches × 3 corruption rounds, exactly as the final fit of [E04](../E04_wave3/README.md). It runs on CPU with one thread, as in its campaign (the local cuDNN GRU crashed at shutdown there).

In [ ]:
if RETRAIN:
    t0 = time.time()
    tfs.train_gru()
    print(f"BiGRU training: {time.time() - t0:.0f} s")

## [T3] Train the short-loss PP-TCN

The 75% component: the residual dilated TCN with the attention/max-pooled player-set branch, trained on all 19 training matches (5 IDSSE × 3 rounds, 14 SkillCorner) with the **state loss doubled on complete live spells shorter than 4 s**. Recipe as frozen in [PLAN.md](PLAN.md): 3 seeds (0/1/2) × 1,500 updates, batch 16, crop 384, AdamW lr 0.002, weight decay 0.01, OneCycle, dropout 0.15, ball masking 0.3.

In [ ]:
if RETRAIN:
    t0 = time.time()
    tfs.train_tcn(TRAIN_DEVICE)
    print(f"PP-TCN training on {TRAIN_DEVICE}: {time.time() - t0:.0f} s")

## [T4] Target inputs (licensed match data)

Builds, from `data/`, everything the pipeline reads about the target match: the parsed tracking caches (`experiments/_cache/atd_full.npy`, `team_dist.npy`), the 5-fps target pack (E04 `prepare_local`), its features (E05 `prepare_features`) and the BiGRU's probabilities on it (the "prior", only the BiGRU columns the final method uses). No target label is read: the target pack's labels are all unknown (−1).

In [ ]:
if RETRAIN:
    t0 = time.time()
    tfs.build_target_inputs()
    print(f"target inputs ready in {time.time() - t0:.0f} s")

# Dataset

## [1] Inputs and identity

Every required artifact must be present. The checkpoint is compared with the hash frozen at selection time: after `RETRAIN` it is a new model and the hashes are expected to differ; without retraining they must match, otherwise the explanations would describe a different model than the submitted one.

In [ ]:
REQUIRED = {
    "target observations": c.base.pack_path("target", "clean"),
    "target features": c.base.CACHE / "features" / "target_clean.npz",
    "BiGRU probabilities (prior)": c.base.CACHE / "prior" / "target_clean.npz",
    "short-loss TCN checkpoint": c.OUT / "models" / "short_tcn.pt",
    **{f"training pack {n}": c.base.pack_path(n, "r0" if n.startswith("J03") else "clean") for n in c.DEV},
}
rows = [{"artifact": k, "path": str(Path(v).relative_to(ROOT)), "present": Path(v).exists(),
         "MB": round(Path(v).stat().st_size / 1e6, 1) if Path(v).exists() else None} for k, v in REQUIRED.items()]
table = pd.DataFrame(rows)
display(table.head(4))
print(f"training packs present: {table.present[4:].sum()} / {len(table) - 4}")
missing = table[~table.present]
if len(missing):
    raise FileNotFoundError("Restore these gitignored artifacts first:\n" + "\n".join(missing.path))

CKPT = REQUIRED["short-loss TCN checkpoint"]
expected_ckpt = DEV["checkpoint_sha256"]["short_tcn"]
actual_ckpt = sha256(CKPT)
CKPT_OK = actual_ckpt == expected_ckpt
print("checkpoint sha256:", actual_ckpt[:16], "| frozen:", expected_ckpt[:16], "| match:", CKPT_OK)
if not CKPT_OK:
    print("retrained checkpoint (differs from the frozen one, as expected)" if RETRAIN else
          "WARNING: checkpoint differs from the frozen selection; results below do not describe the submitted model.")

## [2] Target observations

The target pack holds the 5-fps observation stream (two halves as blocks), the 16-column observation matrix `X` used by the decoder, and the prepared features used by the networks. The target has **no labels**: the cell checks that before anything else.

In [ ]:
data = c.load("target", "clean", with_features=True)
if np.any(data["y"] >= 0) or np.any(data["y25"] >= 0):
    raise ValueError("Target must not contain labels")
obs = c.observations(data)                       # what the models may see
dobs = c.base.decoder_observations(data)         # adds nearest home/away distances for the decoder
X = np.asarray(data["X"], dtype=np.float64)
N5 = len(X)
BLOCKS = [tuple(map(int, b)) for b in data["blocks"]]
BLOCKS25 = [tuple(map(int, b)) for b in data["blocks25"]]
VISIBLE = np.isfinite(X[:, 3]) & (X[:, 3] >= 0.5)
print(f"5-fps samples: {N5} | blocks (5 fps): {BLOCKS} | blocks (25 fps): {BLOCKS25}")
print(f"25-fps frames: {int(data['n_frames'])} | ball visible on {VISIBLE.mean():.1%} of samples")
print("feature arrays:", {k: v.shape for k, v in data["_features"].items() if hasattr(v, "shape")})

def frame_of(i):
    """5-fps sample index -> first 25-fps frame it covers."""
    for (s, e), (b, f) in zip(BLOCKS, BLOCKS25):
        if s <= i < e:
            return b + 5 * (i - s)
    raise ValueError(i)

def to25(values, fill=np.nan):
    """Expand a per-sample (5 fps) array to the 25-fps tactical timeline, exactly like c.expand."""
    values = np.asarray(values)
    out = np.full(int(data["n_frames"]), fill, dtype=np.float64)
    for (s, e), (b, f) in zip(BLOCKS, BLOCKS25):
        out[b:f] = np.repeat(values[s:e], 5)[:f - b]
    return out

## [3] Training possession rate

The decoder never sees a target label. Its only calibration target is the **possession rate** (possessions per minute of graded time) and dead share, averaged per provider over the 19 training matches. The expected number of possessions on the target is that rate times the target's graded minutes.

In [ ]:
stats = c.training_stats(c.DEV)
by_provider = {"IDSSE (5)": c.training_stats(c.IDSSE_DEV), "SkillCorner (14)": c.training_stats(c.SC_DEV)}
graded_min = sum(e - s for s, e in BLOCKS) / FPS5 / 60
rate_tab = pd.DataFrame({**by_provider, "provider-balanced (used)": stats}).T
rate_tab["expected target possessions"] = rate_tab["rate"] * graded_min
display(rate_tab.round(3))
print(f"graded minutes: {graded_min:.1f}")

# Final pipeline

## [4] Short-loss PP-TCN inference

`models.predict` averages three seeds and two mirrored team views. The helper `tcn_forward` below re-implements that loop **per seed** so later sections can inspect seeds and perturb inputs. It must reproduce `models.predict` exactly; the cell asserts this before going on.

In [ ]:
fitted = torch.load(CKPT, weights_only=False, map_location="cpu")
NETS = [net.eval().to("cpu") for net in fitted["nets"]]
MEAN = np.asarray(fitted["mean"], np.float32); SCALE = np.asarray(fitted["scale"], np.float32)
HCFG = models._wave4_config(fitted["config"])
ARR = models._WAVE4._feature_arrays(obs, need_pp=True, need_control=False)
print("config:", {k: fitted["config"][k] for k in ("name", "steps", "ch", "dilations", "seeds", "dropout")})
print("parameters per network:", sum(p.numel() for p in NETS[0].parameters()))

def net_input(start, end, view):
    x, pp, ppm = models._WAVE4._inference_input(ARR, start, end, view, MEAN, SCALE, HCFG)
    return (torch.from_numpy(x[None]), torch.from_numpy(pp[None]).float(), torch.from_numpy(ppm[None]))

def tcn_forward(transform=None, blocks=None):
    """Per-seed p_dead and p_home (view-averaged), shape (3, N5). transform(x, pp, ppm) may perturb inputs."""
    blocks = BLOCKS if blocks is None else blocks
    dead = np.zeros((len(NETS), N5)); home = np.zeros((len(NETS), N5))
    for k, net in enumerate(NETS):
        for start, end in blocks:
            for view in (0, 1):
                x, pp, ppm = net_input(start, end, view)
                if transform is not None:
                    x, pp, ppm = transform(x, pp, ppm)
                logits = net(x, pp, ppm)[0]
                p_dead = torch.sigmoid(logits[0]).numpy(); p_view = torch.sigmoid(logits[1]).numpy()
                dead[k, start:end] += 0.5 * p_dead
                home[k, start:end] += 0.5 * (p_view if view == 0 else 1.0 - p_view)
    return dead, home

def tcn_prediction(dead, home):
    """Combine per-seed arrays exactly like training_models.predict."""
    d = np.clip(dead.mean(0), 0, 1).astype(np.float32); h = np.clip(home.mean(0), 0, 1).astype(np.float32)
    prob = np.empty((N5, 3), np.float32); live = 1.0 - d
    prob[:, 0] = live * h; prob[:, 1] = live * (1.0 - h); prob[:, 2] = d
    tot = prob.sum(1, keepdims=True); np.divide(prob, tot, out=prob, where=tot > 0)
    lt = prob[:, 0] + prob[:, 1]
    return {"probability": prob, "p_dead": prob[:, 2].copy(),
            "p_team": np.divide(prob[:, 0], lt, out=np.full(N5, 0.5, np.float32), where=lt > 1e-8),
            "boundary": np.full(N5, 0.5, np.float32), "has_boundary": False}

t0 = time.time()
tcn = models.predict(fitted, obs)
print(f"models.predict: {time.time() - t0:.1f} s")
SEED_DEAD, SEED_HOME = tcn_forward()
replica = tcn_prediction(SEED_DEAD, SEED_HOME)
assert np.allclose(replica["probability"], tcn["probability"], atol=1e-5), "per-seed replica drifted from models.predict"
print("per-seed replica matches models.predict (max abs diff "
      f"{np.abs(replica['probability'] - tcn['probability']).max():.1e})")

## [5] Fusion with the BiGRU

The BiGRU's probabilities on the target come from [T4] (or from the cached prior when `RETRAIN = False`). They are mixed with the TCN in **joint** H/A/D space and renormalised, using the campaign's own `fusion.combine`.

In [ ]:
with np.load(REQUIRED["BiGRU probabilities (prior)"], allow_pickle=False) as z:
    gru = {"probability": z["hybrid_probability"], "boundary": z["hybrid_boundary"], "has_boundary": False}
assert len(gru["probability"]) == N5
fused = fusion.combine("short_gru25", {"short_tcn": tcn}, None, gru, obs)
gru_res = fusion._source_result(gru, "gru", force_no_boundary=True)
comp = pd.DataFrame({
    "short-TCN": [tcn["p_dead"].mean(), np.mean(tcn["p_team"][tcn["p_dead"] < 0.5])],
    "BiGRU": [gru_res["p_dead"].mean(), np.mean(gru_res["p_team"][gru_res["p_dead"] < 0.5])],
    "fused (75/25)": [fused["p_dead"].mean(), np.mean(fused["p_team"][fused["p_dead"] < 0.5])],
}, index=["mean P(dead)", "mean P(home | live)"])
display(comp.round(3))
agree = np.mean((tcn["p_dead"] > 0.45) == (gru_res["p_dead"] > 0.45))
team_agree = np.mean((tcn["p_team"] > 0.5) == (gru_res["p_team"] > 0.5))
print(f"TCN vs BiGRU agreement: dead/live {agree:.1%} | team (argmax) {team_agree:.1%}")

## [6] Decoding with `dead45`

`decoder.decode` is the frozen decoder. The cell then rebuilds its internal steps (dead mask → team evidence → interval-reset Viterbi → merge) with the same functions, sweeps the switch cost λ over the campaign grid, and asserts that the rebuilt labels at the calibrated λ equal the decoder's output. Section [9] uses these intermediate arrays.

In [ ]:
result = decoder.decode(fused, dobs, DEC, stats)
CODES = result["codes"]
print(f"calibrated lambda {result['lambda']} | target count {result['target_count']:.1f} | decoded possessions {result['actual_count']}")

R = decoder._resolve_config(DEC)
P_DEAD, P_TEAM = decoder._prediction_components(fused, N5)
NB = decoder._normalize_blocks(dobs["blocks"], N5)
NB_ARR = np.asarray(NB, dtype=np.int64).reshape(-1, 2)
MERGE = int(math.ceil(R["merge_s"] * FPS5))

def decode_steps(p_dead, p_team, cfg=R, lam=None, merge=True):
    dead = decoder.dead_mask(p_dead, NB, threshold=cfg["dead_threshold"], min_s=cfg["min_dead_s"], shift=0.0)
    home, away = decoder._team_evidence(p_team, X, dobs, cfg)
    def labels(l, do_merge=merge):
        out = decoder._decode_live_team(home, away, NB, dead, np.empty(0, np.float64), l)
        return decoder._merge_short_runs(out, NB, MERGE) if do_merge else out
    return dead, home, away, labels

DEAD, HOME_EV, AWAY_EV, LABELS = decode_steps(P_DEAD, P_TEAM)
curve = pd.DataFrame({"lambda": decoder.LAMBDA_GRID,
                      "possessions": [decoder._count_possessions(LABELS(l), NB_ARR) for l in decoder.LAMBDA_GRID]})
assert np.array_equal(LABELS(result["lambda"]), CODES), "rebuilt decoder steps drifted from decoder.decode"
PRE_MERGE = LABELS(result["lambda"], do_merge=False)
print("rebuilt dead mask -> evidence -> Viterbi -> merge reproduces decoder.decode")

fig, ax = plt.subplots(1, 2, figsize=(12, 3.4))
ax[0].plot(curve["lambda"], curve["possessions"], color=C2, marker="o", ms=3)
ax[0].axhline(result["target_count"], color=C1, ls="--", lw=1.2, label="target from training rate")
ax[0].axvline(result["lambda"], color=GREY, ls=":", lw=1.2, label=f"chosen λ = {result['lambda']}")
ax[0].set(xscale="log", xlabel="switch cost λ", ylabel="decoded possessions", title="Rate calibration of the switch cost")
ax[0].legend(loc="upper right")
names = ["H", "A", "D"]
share = [np.mean(CODES == k) for k in range(3)]
ax[1].bar(names, share, color=[C1, C2, GREY]); ax[1].set(ylabel="share of graded samples", title="Decoded label shares (full match)")
for a, l in zip(ax, "AB"): panel(a, l)
fig.tight_layout(); plt.show()

## [7] Export and reproducibility check

The labels are expanded from 5 to 25 fps by the campaign's own writer, validated against the official format, and hashed. A match with the locked SHA-256 means this notebook reproduced the submitted CSV **byte for byte**.

In [ ]:
CSV = HERE / "out" / "notebook" / "final_wave5.csv"
csv_sha = old_runner.submission(CSV, CODES, data)
LOCKED = FINAL["artifact"]["sha256"]
REPRODUCED = csv_sha == LOCKED
LAB25_TMP = pm.label_codes(pm.read_label_csv(CSV))
print(f"written: {CSV.relative_to(ROOT)}")
print(f"sha256 {csv_sha[:16]}… | locked {LOCKED[:16]}… | reproduced: {REPRODUCED}")
if not REPRODUCED:
    frozen = pm.label_codes(pm.read_label_csv(ROOT / "submission.csv")) if (ROOT / "submission.csv").exists() else None
    if frozen is not None:
        print(f"frame agreement with the frozen submission: {np.mean(frozen == LAB25_TMP):.4%}")
    print("retrained model: the CSV differs from the frozen submission; compare the public scores below" if RETRAIN else
          "WARNING: output differs from the locked artifact. Check the input caches before trusting any explanation below.")
LAB25 = pm.label_codes(pm.read_label_csv(CSV))

## [8] Public window (diagnostic only)

The official scorer on the 5-minute public GT. These 7,500 frames and 17 possessions were already seen during the campaign; the score is a check that the run is correct, not a model-selection signal. The recorded post-freeze values are spellF1 0.772414 and macroF1 0.902059.

In [ ]:
GT = pm.label_codes(pm.read_label_csv(ROOT / "evaluation" / "gt" / "BAR-ATH_possession_gt_public.csv"))
score = pm.evaluate(GT, LAB25, taus=pm.TAUS, bootstrap_n=0)
rec = FINAL["metrics"]["competition_public"]
display(pd.DataFrame({"this run": [score["spell_f1_mean"], score["macro_f1"], score["n_pred_spells"], score["n_gt_spells"]],
                      "recorded": [rec["spellF1"], rec["macroF1"], rec["predicted_possessions"], rec["gt_possessions"]]},
                     index=["spellF1", "macroF1", "predicted possessions", "GT possessions"]).round(6))
RUNS = pm.valid_runs(GT)
LO, HI = RUNS[0][0], RUNS[-1][1]
def to5(frame):
    """25-fps frame -> 5-fps sample index (inside a block)."""
    for (s, e), (b, f) in zip(BLOCKS, BLOCKS25):
        if b <= frame < f:
            return s + (frame - b) // 5
    raise ValueError(frame)
W5 = slice(to5(LO), to5(HI - 1) + 1)
T_MIN = (np.arange(W5.start, W5.stop) - BLOCKS[0][0]) * 5 / FPS25 / 60 + BLOCKS25[0][0] / FPS25 / 60
print(f"public window: frames {LO}–{HI - 1} = samples {W5.start}–{W5.stop - 1}")

# Explainability

## [9] Anatomy of a decision

Each panel is one pipeline stage on the public window, top to bottom: (A) dead-ball probability of each component against the 0.45 threshold, with the resulting dead mask shaded; (B) $P(\text{home} \mid \text{live})$ from each component; (C) the decoder's team evidence $\ell_H - \ell_A$, with samples where the proximity gate fired marked; (D) labels before the 1-s merge, after it, and the GT.

In [ ]:
dist = np.asarray(dobs["distances"], dtype=np.float64)
known = np.isfinite(dist).all(1) & (np.nan_to_num(dist, nan=-1.0) >= 0).all(1)
GATE = VISIBLE & known & (np.where(known, dist.min(1), np.inf) <= R["gate_r"])   # same condition as decoder._team_evidence
seg = lambda a: np.asarray(a)[W5]
fig, ax = plt.subplots(4, 1, figsize=(12, 9), sharex=True, gridspec_kw=dict(height_ratios=[1, 1, 1, 0.9]))
ax[0].fill_between(T_MIN, 0, 1, where=seg(DEAD), color=GREY, alpha=0.35, step="mid", lw=0, label="dead mask")
ax[0].plot(T_MIN, seg(tcn["p_dead"]), color=TEAL, lw=0.9, label="short-TCN")
ax[0].plot(T_MIN, seg(gru_res["p_dead"]), color=C2, lw=0.9, ls="--", label="BiGRU")
ax[0].plot(T_MIN, seg(P_DEAD), color=C1, lw=1.3, label="fused")
ax[0].axhline(R["dead_threshold"], color=WARM, ls=":", lw=1); ax[0].set(ylim=(0, 1), ylabel="P(dead)", title="Dead-ball probability and mask")
ax[1].plot(T_MIN, seg(tcn["p_team"]), color=TEAL, lw=0.9, label="short-TCN")
ax[1].plot(T_MIN, seg(gru_res["p_team"]), color=C2, lw=0.9, ls="--", label="BiGRU")
ax[1].plot(T_MIN, seg(P_TEAM), color=C1, lw=1.3, label="fused")
ax[1].axhline(0.5, color=WARM, ls=":", lw=1); ax[1].set(ylim=(0, 1), ylabel="P(home | live)", title="Team probability")
ev = seg(HOME_EV - AWAY_EV)
ax[2].plot(T_MIN, ev, color=C2, lw=0.9)
ax[2].scatter(T_MIN[seg(GATE)], ev[seg(GATE)], s=6, color=C1, zorder=3, label="proximity gate active")
ax[2].axhline(0, color=WARM, ls=":", lw=1); ax[2].set(ylabel="ℓ_H − ℓ_A", title="Decoder team evidence")
gt5 = np.array([GT[frame_of(i)] for i in range(W5.start, W5.stop)])
for y, lab, name in ((2, seg(PRE_MERGE), "Viterbi"), (1, seg(CODES), "after merge"), (0, np.where(gt5 > 2, 3, gt5), "GT")):
    for k, col in enumerate([C1, C2, GREY]):
        ax[3].fill_between(T_MIN, y - 0.4, y + 0.4, where=lab == k, color=col, step="mid", lw=0)
ax[3].set(yticks=[0, 1, 2], yticklabels=["GT", "after merge", "Viterbi"], xlabel="tactical video time (min)", title="Labels")
for a in ax[:3]: a.legend(loc="center left", bbox_to_anchor=(1.0, 0.5))
for a, l in zip(ax, "ABCD"): panel(a, l)
fig.tight_layout(); plt.show()
print(f"proximity gate active on {GATE.mean():.1%} of samples (full match), {seg(GATE).mean():.1%} in the public window")

**How to read it.** The decoder never thresholds $P(\text{home})$ frame by frame. A possession change happens only where the accumulated evidence in panel C outweighs λ. Short flickers in panels A–B that do not survive into panel D were absorbed by λ or by the 1-s merge. Compare the Viterbi and after-merge rows to see which possessions the merge removed.

> Findings after the rerun: _to be written_.

## [10] Counterfactual stage ablation

Each row changes **one** stage and keeps everything else frozen. The full-match columns need no labels. The public columns are diagnostics on 17 possessions. This table explains the frozen choice; it is not used to change it. Rows:

- decoder variants that were in the wave5 grid (`wave4`, `postmerge`, `live_reset`, `dead1`, `dead55`, `no_merge`),
- proximity gate off ($r = 0$),
- BiGRU weight 0 (TCN only), 0.5 and 1 (BiGRU only),
- switch cost fixed at ½λ and 2λ instead of rate calibration.

In [ ]:
def decode_with(prediction, cfg=None, lam_scale=None):
    cfg = dict(DEC) if cfg is None else cfg
    if lam_scale is None:
        return decoder.decode(prediction, dobs, cfg, stats)["codes"]
    pdead, pteam = decoder._prediction_components(prediction, N5)
    *_, labels = decode_steps(pdead, pteam, decoder._resolve_config(cfg))
    return labels(result["lambda"] * lam_scale)

def mix(w):
    return fusion._result(fusion._joint_mix(tcn["probability"], gru["probability"], w), np.full(N5, 0.5, np.float32), False)

def summarise(name, codes5):
    lab = to25(codes5, fill=3).astype(np.int8)
    s = pm.evaluate(GT, lab, taus=pm.TAUS, bootstrap_n=0)
    spells = pm.possession_spells(lab, [(b, f) for b, f in BLOCKS25])
    dur = np.array([(e - a) / FPS25 for a, e, _ in spells])
    return {"variant": name, "possessions": len(spells), "median poss. (s)": np.median(dur),
            "share < 4 s": np.mean(dur < 4), "D share": np.mean(codes5 == 2),
            "public spellF1": s["spell_f1_mean"], "public macroF1": s["macro_f1"]}

variants = [("final (dead45)", CODES)]
for row in decoder.CONFIGS:
    if row["name"] in ("wave4", "postmerge", "live_reset", "dead1", "dead55", "no_merge"):
        variants.append((f"decoder {row['name']}", decode_with(fused, dict(row))))
variants.append(("proximity gate off", decode_with(fused, {**DEC, "gate_r": 0.0})))
for w in (0.0, 0.5, 1.0):
    variants.append((f"BiGRU weight {w}", decode_with(mix(w))))
for k in (0.5, 2.0):
    variants.append((f"λ fixed at {k}×", decode_with(fused, lam_scale=k)))
ABL = pd.DataFrame([summarise(n, v) for n, v in variants]).set_index("variant")
display(ABL.round(3))

fig, ax = plt.subplots(1, 2, figsize=(12, 4.2), sharey=True)
y = np.arange(len(ABL))
ax[0].barh(y, ABL["possessions"], color=[C1 if i == 0 else C2 for i in y]); ax[0].axvline(result["target_count"], color=WARM, ls="--", lw=1)
ax[0].set(yticks=y, yticklabels=ABL.index, xlabel="full-match possessions (dashed = training-rate target)", title="Segmentation (no labels needed)")
ax[1].barh(y, ABL["public spellF1"], color=[C1 if i == 0 else GREY for i in y])
ax[1].set(xlabel="public spellF1 (diagnostic)", title="Public window, 17 possessions")
ax[0].invert_yaxis()
for a, l in zip(ax, "AB"): panel(a, l)
fig.tight_layout(); plt.show()

**How to read it.** A variant that changes the full-match possession count a lot (panel A) has a large effect on segmentation, whether or not the public score moves. One missed possession moves public spellF1 by about 0.06, so public differences below that are within noise. The development evidence for these choices is in [`results/wave5_development.json`](results/wave5_development.json) (registry [E06](CARD.md)).

> Findings after the rerun: _to be written_.

## [11] Which inputs drive the network: Integrated Gradients

Integrated Gradients (IG) attributes a logit to the inputs by integrating gradients along a straight path from a baseline to the actual input:

$$
\mathrm{IG}_i(x) = (x_i - x'_i) \int_0^1 \frac{\partial F\big(x' + \alpha (x - x')\big)}{\partial x_i}\, d\alpha .
$$

The baseline $x'$ is all zeros. For the standardised team-shape channels that is the training mean; for ball channels and the player set it is the "missing" value the model already sees when tracking drops out. $F$ is the sum of one logit (dead, or "home has the ball" from the home view) over the interior of a crop, so edge effects are excluded. The integral uses 64 Riemann steps. Completeness ($\sum_i \mathrm{IG}_i \approx F(x) - F(x')$) is checked. Attributions are computed on 16 seeded random crops of 512 samples (≈ 102 s each) across the graded match, for all three seeds.

In [ ]:
CH_NAMES = list(FEATS.PLAYER_FEATS) + ["features valid"] + list(FEATS.BALL_FEATS)
assert len(CH_NAMES) == 29
RF = 1 + 4 * sum(fitted["config"]["dilations"])          # receptive field in samples (kernel 5)
HALF_RF = RF // 2
CROP, STEPS, N_CROPS = 512, 64, 16
rng = np.random.default_rng(20261117)
crops = []
while len(crops) < N_CROPS:
    s, e = BLOCKS[rng.integers(len(BLOCKS))]
    a = int(rng.integers(s, e - CROP)); crops.append((a, a + CROP))

def integrated_gradients(net, x, pp, ppm, head):
    alphas = torch.linspace(1.0 / STEPS, 1.0, STEPS).view(-1, 1, 1)
    xs = (alphas * x).requires_grad_(True)
    pps = (alphas.view(-1, 1, 1, 1) * pp).requires_grad_(True)
    with torch.enable_grad():
        out = net(xs, pps, ppm.expand(STEPS, -1, -1))[:, head, HALF_RF:-HALF_RF].sum()
        gx, gp = torch.autograd.grad(out, (xs, pps))
        f_x = net(x, pp, ppm)[0, head, HALF_RF:-HALF_RF].sum()
        f_0 = net(torch.zeros_like(x), torch.zeros_like(pp), ppm)[0, head, HALF_RF:-HALF_RF].sum()
    ig_x = (x[0] * gx.mean(0)).numpy(); ig_p = (pp[0] * gp.mean(0)).numpy()
    return ig_x, ig_p, float(f_x - f_0)

rows, comp_err = [], []
for k, net in enumerate(NETS):
    for a, b in crops:
        x, pp, ppm = net_input(a, b, view=0)
        for head, hname in ((0, "dead"), (1, "team")):
            ig_x, ig_p, delta = integrated_gradients(net, x, pp, ppm, head)
            total = ig_x.sum() + ig_p.sum()
            comp_err.append(abs(total - delta) / (abs(delta) + 1e-6))
            ch = np.abs(ig_x).sum(1); pf = np.abs(ig_p).sum((0, 1))
            rows += [{"seed": k, "head": hname, "group": "frame channel", "input": n, "abs_ig": v} for n, v in zip(CH_NAMES, ch)]
            rows += [{"seed": k, "head": hname, "group": "player set", "input": "player: " + n, "abs_ig": v} for n, v in zip(FEATS.PP_FEATS, pf)]
IG = pd.DataFrame(rows)
print(f"receptive field {RF} samples ({RF / FPS5:.1f} s) | IG completeness error median {np.median(comp_err):.1%}, P90 {np.percentile(comp_err, 90):.1%}")

share = IG.groupby(["head", "seed", "input"]).abs_ig.sum()
share = share / share.groupby(level=[0, 1]).transform("sum")
S = share.groupby(level=[0, 2]).agg(["mean", "std"])
fig, ax = plt.subplots(1, 2, figsize=(13, 8.5), sharey=True)
order = S.loc["team"].sort_values("mean").index
for a, head, col in ((ax[0], "dead", C1), (ax[1], "team", C1)):
    m = S.loc[head].reindex(order)
    colors = [TEAL if i.startswith("player: ") else col for i in order]
    a.barh(range(len(order)), m["mean"], xerr=m["std"], color=colors, error_kw=dict(lw=0.6, capsize=1.5))
    a.set(yticks=range(len(order)), yticklabels=order, xlabel="share of total |IG| (mean ± sd over seeds)",
          title=f"{'Dead-ball' if head == 'dead' else 'Team'} logit")
fig.legend(handles=[Patch(color=C1, label="frame channel"), Patch(color=TEAL, label="player-set feature")], loc="lower center", ncol=2)
for a, l in zip(ax, "AB"): panel(a, l)
fig.tight_layout(rect=(0, 0.03, 1, 1)); plt.show()

grp = IG.assign(block=np.where(IG.input.str.startswith("player: "), "player set",
                np.where(IG.input.isin(FEATS.BALL_FEATS), "ball channels", np.where(IG.input == "features valid", "validity", "team shape"))))
g = grp.groupby(["head", "seed", "block"]).abs_ig.sum(); g = g / g.groupby(level=[0, 1]).transform("sum")
display(g.groupby(level=[0, 2]).agg(["mean", "std"]).unstack(0).round(3))
rank = share.unstack(level=[0, 1])
cons = {h: np.mean([rank[(h, i)].rank().corr(rank[(h, j)].rank()) for i in range(3) for j in range(i + 1, 3)]) for h in ("dead", "team")}
print("seed-to-seed rank correlation of attributions:", {h: round(v, 3) for h, v in cons.items()})

**How to read it.** The bars rank inputs by their share of total absolute attribution, so they show influence, not direction. The block table shows how much of each decision rests on team shape, ball channels and the player set. A high seed-to-seed rank correlation means the explanation is a property of the method, not of one random initialisation. IG explains **this model**, not the true cause of possession: correlated inputs (Section [9] of E00) can share credit.

> Findings after the rerun: _to be written_.

## [12] How far in time the network looks

The TCN's theoretical receptive field is $1 + 4\sum d = 129$ samples (±12.8 s) and is **centred**, so the model is offline: it uses the future as well as the past. The cell measures the *effective* receptive field: the input-gradient magnitude of the team and dead logits at a centre sample, as a function of time offset, averaged over the IG crops and seeds. The BiGRU, which sees the whole window in both directions, is not included.

In [ ]:
prof = {0: np.zeros(RF), 1: np.zeros(RF)}
for net in NETS:
    for a, b in crops:
        x, pp, ppm = net_input(a, b, view=0)
        centre = CROP // 2
        for head in (0, 1):
            xg = x.clone().requires_grad_(True); pg = pp.clone().requires_grad_(True)
            with torch.enable_grad():
                g, gp = torch.autograd.grad(net(xg, pg, ppm)[0, head, centre], (xg, pg))
            mag = g[0].abs().sum(0).numpy() + gp[0].abs().sum((1, 2)).numpy()
            prof[head] += mag[centre - HALF_RF:centre + HALF_RF + 1]
off = (np.arange(RF) - HALF_RF) / FPS5
fig, ax = plt.subplots(1, 2, figsize=(12, 3.4))
for head, col, name in ((0, GREY, "dead logit"), (1, C1, "team logit")):
    p = prof[head] / prof[head].sum()
    ax[0].plot(off, p, color=col, lw=1.4, label=name)
    order = np.argsort(np.abs(off)); cum = np.cumsum(p[order])
    ax[1].plot(np.abs(off)[order], cum, color=col, lw=1.4, label=name)
    print(f"{name}: 50% of gradient mass within ±{np.abs(off)[order][np.searchsorted(cum, 0.5)]:.1f} s, 90% within ±{np.abs(off)[order][np.searchsorted(cum, 0.9)]:.1f} s")
ax[0].axvline(0, color=WARM, ls=":", lw=1); ax[0].set(xlabel="offset from decided sample (s)", ylabel="share of |∂logit/∂input|", title="Effective temporal receptive field")
ax[1].set(xlabel="|offset| (s)", ylabel="cumulative share", ylim=(0, 1.02), title="How much context is used")
ax[0].legend(loc="upper right")
for a, l in zip(ax, "AB"): panel(a, l)
fig.tight_layout(); plt.show()

**How to read it.** A profile concentrated near 0 s means the TCN decides mostly from the local moment, leaving long-range consistency to the decoder. Mass far from 0 on both sides means the network itself bridges ball-missing gaps using context before **and after** the gap. That is what EDA §[2] showed is needed (median gap 10.7 s).

> Findings after the rerun: _to be written_.

## [13] Which players the set encoder attends to

The per-player branch encodes up to 48 player vectors, pools them with a softmax attention plus a masked max, and adds the result to the temporal stream. The cell reads the attention weights directly from the network on the IG crops (home view) and relates them to each player's distance to the ball. The max-pooling path is not visible in these weights, so attention is only part of the branch's behaviour.

In [ ]:
BALLD = list(FEATS.PP_FEATS).index("ball distance"); SAME = list(FEATS.PP_FEATS).index("same team")
by_rank = {True: np.zeros(10), False: np.zeros(10)}; cnt = {True: 0, False: 0}
top_share, own_share = {True: [], False: []}, {True: [], False: []}
for net in NETS:
    for a, b in crops:
        x, pp, ppm = net_input(a, b, view=0)
        enc = net.ppnet(pp).masked_fill(~ppm.unsqueeze(-1), 0.0)
        att = torch.softmax(net.ppatt(enc).squeeze(-1).masked_fill(~ppm, -1e4), dim=-1)[0].numpy()
        mask = ppm[0].numpy(); dist = pp[0, ..., BALLD].numpy(); same = pp[0, ..., SAME].numpy()
        vis = VISIBLE[a:b]
        for t in range(att.shape[0]):
            m = mask[t]
            if m.sum() < 10: continue
            order = np.argsort(np.where(m, dist[t], np.inf))[:10]
            v = bool(vis[t]); by_rank[v] += att[t, order]; cnt[v] += 1
            top_share[v].append(att[t, order[0]] * m.sum())        # >1 means above uniform
            own_share[v].append(att[t, m & (same[t] > 0.5)].sum())
fig, ax = plt.subplots(1, 2, figsize=(12, 3.4))
for v, col, name in ((True, C1, "ball visible"), (False, C2, "ball missing")):
    if cnt[v]:
        ax[0].plot(np.arange(1, 11), by_rank[v] / cnt[v], color=col, marker="o", ms=3, label=name)
ax[0].set(xlabel="player rank by distance to ball (1 = nearest)", ylabel="mean attention weight", title="Attention vs ball distance")
ax[0].legend()
data_box = [np.asarray(top_share[v]) for v in (True, False) if len(top_share[v])]
ax[1].boxplot(data_box, tick_labels=[n for v, n in ((True, "ball visible"), (False, "ball missing")) if len(top_share[v])], showfliers=False)  # noqa
ax[1].axhline(1, color=WARM, ls="--", lw=1); ax[1].set(ylabel="attention on nearest player ÷ uniform", title="Focus on the nearest player")
for a, l in zip(ax, "AB"): panel(a, l)
fig.tight_layout(); plt.show()
for v, name in ((True, "visible"), (False, "missing")):
    if cnt[v]:
        print(f"ball {name}: attention on own (home-view) team {np.mean(own_share[v]):.1%} | nearest-player focus ×{np.median(top_share[v]):.2f} uniform")

**How to read it.** If the curve falls steeply with rank while the ball is visible, the branch is acting as a learned "who is closest" detector. If the gap between the visible and missing curves is small, the branch relies on player configuration rather than the ball, which is what lets it keep working through tracking gaps.

> Findings after the rerun: _to be written_.

## [14] Feature-group occlusion

Each input group is replaced by its zero baseline over the whole match (as in [11]), and the full pipeline (TCN → fusion → `dead45`) is rerun. Groups: team-shape channels, the validity flag, ball channels, and the whole player-set branch (all players masked). The table reports how far probabilities move and what happens to the segmentation.

This perturbs the **standardised inputs**; it is not a physical simulation. For ball loss in the raw data, see the paired ball-drop stress test in [E06 CARD](CARD.md).

In [ ]:
PLAYER_CH, VALID_CH, BALL_CH = list(range(23)), [23], list(range(24, 29))
def zero_channels(idx):
    def t(x, pp, ppm):
        x = x.clone(); x[:, idx] = 0.0
        return x, pp, ppm
    return t
GROUPS = {"team shape (23 ch)": zero_channels(PLAYER_CH), "validity flag": zero_channels(VALID_CH),
          "ball channels (5 ch)": zero_channels(BALL_CH),
          "player set (all masked)": lambda x, pp, ppm: (x, torch.zeros_like(pp), torch.zeros_like(ppm))}
rows = []
base_poss = decoder._count_possessions(CODES, NB_ARR)
for name, tf in GROUPS.items():
    d, h = tcn_forward(tf)
    pred = fusion.combine("short_gru25", {"short_tcn": tcn_prediction(d, h)}, None, gru, obs)
    codes = decoder.decode(pred, dobs, DEC, stats)["codes"]
    live = P_DEAD < 0.5
    rows.append({"occluded": name,
                 "mean |Δ P(dead)|": np.mean(np.abs(pred["p_dead"] - P_DEAD)),
                 "mean |Δ P(home|live)|": np.mean(np.abs(pred["p_team"] - P_TEAM)[live]),
                 "labels changed": np.mean(codes != CODES),
                 "possessions": decoder._count_possessions(codes, NB_ARR),
                 "public spellF1": pm.evaluate(GT, to25(codes, 3).astype(np.int8), taus=pm.TAUS, bootstrap_n=0)["spell_f1_mean"]})
OCC = pd.DataFrame(rows).set_index("occluded")
print(f"reference: {base_poss} possessions, public spellF1 {score['spell_f1_mean']:.3f}")
display(OCC.round(3))
fig, ax = plt.subplots(figsize=(7, 3))
ax.barh(OCC.index, OCC["labels changed"], color=C1); ax.invert_yaxis()
ax.set(xlabel="share of samples whose final label changes", title="Decision dependence on each input group")
fig.tight_layout(); plt.show()

**How to read it.** "Labels changed" measures how much the final output depends on a group once the BiGRU (25%) and the decoder have had their say. A group with large probability shifts but few label changes is redundant with other inputs; a group whose removal collapses the possession count is load-bearing.

> Findings after the rerun: _to be written_.

## [15] Where the three seeds disagree

The three seeds share data and recipe and differ only in initialisation and batch order. Their spread is a cheap measure of model uncertainty (epistemic, partial). The cell compares the spread near decoded possession changes with the spread elsewhere.

In [ ]:
p_home_seed = SEED_HOME                       # (3, N5) view-averaged P(home | live) per seed
sd_team = p_home_seed.std(0); sd_dead = SEED_DEAD.std(0)
change = np.zeros(N5, bool)
for s, e in NB:
    idx = np.flatnonzero(CODES[s + 1:e] != CODES[s:e - 1]) + s + 1
    for i in idx: change[max(s, i - 2 * FPS5):min(e, i + 2 * FPS5)] = True
tab = pd.DataFrame({"within ±2 s of a decoded change": [sd_team[change].mean(), sd_dead[change].mean()],
                    "elsewhere": [sd_team[~change].mean(), sd_dead[~change].mean()]}, index=["sd P(home)", "sd P(dead)"])
display(tab.round(4))
fig, ax = plt.subplots(2, 1, figsize=(12, 4.6), sharex=True)
for k, col in zip(range(3), (C1, C2, TEAL)):
    ax[0].plot(T_MIN, seg(p_home_seed[k]), color=col, lw=0.8, label=f"seed {k}")
ax[0].set(ylim=(0, 1), ylabel="P(home)", title="Per-seed team probability (public window)"); ax[0].legend(loc="center left", bbox_to_anchor=(1, 0.5))
ax[1].fill_between(T_MIN, seg(sd_team), color=C1, alpha=0.4, step="mid"); ax[1].plot(T_MIN, seg(sd_team), color=C2, lw=0.8)
ax[1].set(ylabel="sd across seeds", xlabel="tactical video time (min)", title="Seed disagreement")
for a, l in zip(ax, "AB"): panel(a, l)
fig.tight_layout(); plt.show()

**How to read it.** Higher disagreement near decoded changes means the boundaries themselves are the uncertain part, which matches the short-possession weakness recorded in [E06 CARD](CARD.md) (short-spell recall < 25% at IoU 0.75). Seed spread does not capture data uncertainty (for example a wrongly tracked ball), so low spread is not proof of correctness.

> Findings after the rerun: _to be written_.

## [16] Error anatomy on the public window

GT possessions are matched to predicted ones exactly as the scorer does (same team, greedy by IoU, one-to-one). For every GT possession the table shows its best IoU, whether it was matched at 0.5, its length, and what the model saw during it (ball visibility, mean fused $P(\text{home})$, seed disagreement). This turns the public score into specific, inspectable failures. These 17 possessions are not used to change anything.

In [ ]:
gt_sp = pm.possession_spells(GT, RUNS); pr_sp = pm.possession_spells(LAB25, RUNS)
ov = pm.spell_overlaps(gt_sp, pr_sp)
best = {i: 0.0 for i in range(len(gt_sp))}
for iou, i, j in ov: best[i] = max(best[i], iou)
m05 = pm.matched_gt_flags(ov, len(gt_sp), 0.5)
vis25 = to25(VISIBLE.astype(float)); pt25 = to25(P_TEAM); sd25 = to25(sd_team)
rows = []
for i, (a, b, k) in enumerate(gt_sp):
    rows.append({"GT team": "HA"[k], "start (min)": a / FPS25 / 60, "duration (s)": (b - a) / FPS25,
                 "best IoU": best[i], "matched @0.5": bool(m05[i]), "ball visible": np.nanmean(vis25[a:b]),
                 "mean P(home)": np.nanmean(pt25[a:b]), "seed sd": np.nanmean(sd25[a:b])})
ERR = pd.DataFrame(rows)
display(ERR.round(3))
print(f"matched at IoU 0.5: {m05.sum()} / {len(gt_sp)} | predicted possessions in window: {len(pr_sp)}")
print("mean duration (s): matched", round(ERR[ERR['matched @0.5']]['duration (s)'].mean(), 1),
      "| missed", round(ERR[~ERR['matched @0.5']]['duration (s)'].mean(), 1))

cm = pm.confusion(GT, LAB25)[:3, :3]
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6), gridspec_kw=dict(width_ratios=[1.5, 1]))
ax[0].scatter(ERR["duration (s)"], ERR["best IoU"], c=[C1 if t == "H" else C2 for t in ERR["GT team"]],
              s=30 + 60 * ERR["ball visible"].fillna(0), edgecolor=INK, lw=0.4)
ax[0].axhline(0.5, color=WARM, ls="--", lw=1); ax[0].set(xscale="log", xlabel="GT possession duration (s)", ylabel="best IoU with a prediction",
          title="Which possessions are recovered (size = ball visibility)")
norm = cm / cm.sum(1, keepdims=True)
ax[1].imshow(norm, cmap="Greys", vmin=0, vmax=1)
for i in range(3):
    for j in range(3):
        ax[1].text(j, i, f"{cm[i, j]}\n{norm[i, j]:.0%}", ha="center", va="center", fontsize=8, color="white" if norm[i, j] > 0.5 else INK)
ax[1].set(xticks=range(3), yticks=range(3), xticklabels=list("HAD"), yticklabels=list("HAD"), xlabel="predicted", ylabel="GT", title="Frame confusion")
for a, l in zip(ax, "AB"): panel(a, l)
fig.tight_layout(); plt.show()

**How to read it.** Missed possessions that are short, low on ball visibility, or high in seed disagreement point to the documented failure modes: short spells and ball loss. A missed possession with high confidence for the wrong team points to a tracking or labelling disagreement worth checking in the video. The frame confusion shows whether errors are team swaps (H↔A) or late or missed stoppages (D→H/A).

> Findings after the rerun: _to be written_.

# Summary

What this notebook establishes, once rerun:

1. **Reproduction.** Section [7] states whether the run reproduced the locked CSV (`cd59fb90…`). Explanations are only valid for the submitted predictions when it did.
2. **Decision path.** Sections [6] and [9] show that the output is the result of a calibrated segmentation, not per-frame argmax: dead mask → evidence → interval-reset Viterbi at a rate-matched λ → 1-s merge.
3. **What matters.** Sections [10] and [14] rank stages and input groups by how much they change the output; [11] and [13] show which signals the network itself relies on.
4. **Where it fails.** Sections [15] and [16] connect uncertainty and public-window errors to the known weaknesses: short possessions and ball loss.

Limits that hold whatever the rerun shows:
- Attributions explain the model, not football.
- Occlusion perturbs standardised inputs, not the raw data.
- The public window has only 17 possessions.
- The full-match (private) score is unknown.
- Nothing here changes the frozen method.

Method and evidence: [`docs/FINAL_METHOD.md`](../../docs/FINAL_METHOD.md), registry [E06](CARD.md), [`RESULTS.md`](RESULTS.md).